# 13 - GAEZ Rain-fed Audit Columns for All Ten Crops (~9 km)

Notebook 07 switched the original four crops to the irrigated scenario (`HILM`) and kept their rain-fed values (`HRLM`) as
audit columns `gaez_index_<crop>_rainfed` / `gaez_class_<crop>_rainfed`. The six crops added by notebook 08 were only ever
downloaded under `HILM`, so they had no rain-fed counterpart. This notebook downloads the rain-fed scenario for those six
crops (same period, climate source and input level) and adds their audit columns with the same zonal statistics as
notebooks 07 and 08. The 1 km rain-fed columns of all ten crops come from notebook 12. Re-running is safe: crops that
already have their audit columns are skipped.

In [1]:
# add scripts/download to sys.path so we can `from common import ...`
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Download the six added crops under HRLM (rain-fed, high input)

In [2]:
from common import RAW_DIR, PROCESSED_DIR, ensure_dir, get_logger, download_file

log = get_logger("gaez_rainfed_audit")

BASE_URL_TMPL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/{variable}/"
GAEZ_DIR = RAW_DIR / "gaez_v5"
ensure_dir(GAEZ_DIR)

PERIOD = "HP0120"
CLIMATE_SOURCE = "AGERA5"
SSP = "HIST"
WATER_INPUT_RAINFED = "HRLM"  # rain-fed, high input level of management

ADDED_CROP_CODES = {
    "oilpalm": "OLP", "rubber": "RUB", "sugarcane": "SUC",
    "coffee": "COF", "cacao": "COC", "coconut": "CON",
}
VARIABLES = {"RES05-SIX": "suitability_class", "RES05-SXX": "suitability_index"}

rainfed_paths = {}
for crop, code in ADDED_CROP_CODES.items():
    for variable, label in VARIABLES.items():
        dest = GAEZ_DIR / f"{label}_{crop}_{code}_rainfed.tif"
        fname = f"GAEZ-V5.{variable}.{PERIOD}.{CLIMATE_SOURCE}.{SSP}.{code}.{WATER_INPUT_RAINFED}.tif"
        download_file(BASE_URL_TMPL.format(variable=variable) + fname, dest)
        rainfed_paths[(crop, label)] = dest

print("Done -", len(rainfed_paths), "rain-fed rasters ready in", GAEZ_DIR)

suitability_class_oilpalm_OLP_rainfed.tif:   0%|          | 0.00/338k [00:00<?, ?B/s]

suitability_class_oilpalm_OLP_rainfed.tif:  19%|█▉        | 65.5k/338k [00:00<00:03, 89.6kB/s]

suitability_class_oilpalm_OLP_rainfed.tif:  39%|███▉      | 131k/338k [00:01<00:01, 111kB/s]  

suitability_class_oilpalm_OLP_rainfed.tif:  58%|█████▊    | 197k/338k [00:01<00:00, 148kB/s]

suitability_class_oilpalm_OLP_rainfed.tif:  78%|███████▊  | 262k/338k [00:01<00:00, 198kB/s]

suitability_class_oilpalm_OLP_rainfed.tif: 100%|██████████| 338k/338k [00:01<00:00, 193kB/s]

... (107 outputs left out)

suitability_index_coconut_CON_rainfed.tif:  90%|████████▉ | 918k/1.02M [00:02<00:00, 889kB/s]

suitability_index_coconut_CON_rainfed.tif: 100%|██████████| 1.02M/1.02M [00:02<00:00, 433kB/s]


[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_coconut_CON_rainfed.tif


Done - 12 rain-fed rasters ready in D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5


## 2. Zonal statistics onto the existing 1 km grid (same method as notebooks 07 and 08)

In [3]:
import logging

import geopandas as gpd
from rasterstats import zonal_stats

logging.getLogger("rasterio._env").setLevel(logging.ERROR)  # silence repeated GDAL_DATA noise, not an error

GRID_PATH = PROCESSED_DIR / "grid" / "grid_1km.gpkg"
grid = gpd.read_file(GRID_PATH)
grid_4326 = grid.to_crs("EPSG:4326")
print(f"{len(grid)} grid cells loaded")

rainfed_values = {}
for crop in ADDED_CROP_CODES:
    class_stats = zonal_stats(grid_4326.geometry, str(rainfed_paths[(crop, "suitability_class")]),
                              stats=["majority"], all_touched=True)
    index_stats = zonal_stats(grid_4326.geometry, str(rainfed_paths[(crop, "suitability_index")]),
                              stats=["mean"], all_touched=True)
    rainfed_values[(crop, "class")] = [s["majority"] for s in class_stats]
    rainfed_values[(crop, "index")] = [s["mean"] for s in index_stats]
    n_valid = sum(v is not None for v in rainfed_values[(crop, "index")])
    print(f"{crop}: {n_valid}/{len(grid)} cells got a value")

9333 grid cells loaded


oilpalm: 9333/9333 cells got a value


rubber: 9333/9333 cells got a value


sugarcane: 9333/9333 cells got a value


coffee: 9333/9333 cells got a value


cacao: 9333/9333 cells got a value


coconut: 9333/9333 cells got a value


## 3. Add the audit columns to `tabular_features.csv` (idempotent)

In [4]:
import pandas as pd

FEATURES_PATH = PROCESSED_DIR / "features" / "tabular_features.csv"
df = pd.read_csv(FEATURES_PATH)
assert len(df) == len(grid), "row count mismatch between features table and grid"
assert (df["grid_id"].to_numpy() == grid["grid_id"].to_numpy()).all(), "row order differs from the grid"

added = []
for crop in ADDED_CROP_CODES:
    if f"gaez_index_{crop}_rainfed" in df.columns:
        print(f"{crop}: audit columns already present - skipping")
        continue
    df[f"gaez_class_{crop}_rainfed"] = rainfed_values[(crop, "class")]
    df[f"gaez_index_{crop}_rainfed"] = rainfed_values[(crop, "index")]
    added.append(crop)

df.to_csv(FEATURES_PATH, index=False)
print(f"Wrote {FEATURES_PATH} - {len(df)} rows, {len(df.columns)} columns")
print("Crops added this run:", added)

C:\Users\Axioo Pongo\AppData\Local\Temp\ipykernel_21388\2008599022.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"gaez_class_{crop}_rainfed"] = rainfed_values[(crop, "class")]
C:\Users\Axioo Pongo\AppData\Local\Temp\ipykernel_21388\2008599022.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"gaez_index_{crop}_rainfed"] = rainfed_values[(crop, "index")]


Wrote D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\processed\features\tabular_features.csv - 9333 rows, 103 columns
Crops added this run: ['oilpalm', 'rubber', 'sugarcane', 'coffee', 'cacao', 'coconut']


## 4. Check: irrigated vs rain-fed for every crop

In [5]:
ALL_CROPS = ["rice", "maize", "cassava", "soybean", "oilpalm", "rubber", "sugarcane", "coffee", "cacao", "coconut"]
rows = []
for c in ALL_CROPS:
    irr, rf = df[f"gaez_index_{c}"], df[f"gaez_index_{c}_rainfed"]
    rows.append({"crop": c, "mean irrigated": irr.mean(), "mean rain-fed": rf.mean(),
                 "share of cells where rain-fed > irrigated": (rf > irr + 1).mean(),
                 "suitable cells irrigated": int((irr >= 5000).sum()), "suitable cells rain-fed": int((rf >= 5000).sum())})
pd.DataFrame(rows).set_index("crop").round(3)

,mean irrigated,mean rain-fed,share of cells where rain-fed > irrigated,suitable cells irrigated,suitable cells rain-fed
crop,,,,,
rice,3038.963,2565.968,0.000,2001,1640
maize,1314.575,1287.305,0.000,0,0
cassava,3959.674,3959.911,0.001,3154,3154
soybean,1735.126,1597.531,0.003,87,0
oilpalm,4394.978,4395.263,0.001,3957,3957
rubber,2254.936,2255.197,0.001,0,0
sugarcane,2344.300,2344.458,0.001,669,669
coffee,3557.059,3557.318,0.001,2660,2660
cacao,3012.629,3012.801,0.001,1297,1297


## Notes

- Every crop now has its rain-fed (`HRLM`) audit pair at ~9 km: the original four from notebook 07, the six added crops
  from this notebook. Notebook 12 provides the same pair at 1 km.
- The audit columns are not used by the recommendation rule or by the models; the dashboard compares the two scenarios on
  the 'Exploratory Analysis' page.